# Posterior uncertainty for integrated squared density

This notebook evaluates PPB credible intervals under a standard Laplace density, using partitions selected by GEB. Integrated squared density is computed analytically for each posterior draw. Completed coverage replications are skipped on rerun.

Run `02_posterior_uncertainty_dpmm.Rmd` first to generate the DPMM results and the shared dataset used in the posterior comparison figure.


## Configuration

In [ ]:
DIMENSION = 1
N_REPETITIONS = 1_000
N_TRAIN = 5_000
POSTERIOR_DRAWS = 1_000
TRUE_ISD = 0.25  # exact: integral (exp(-abs(z))/2)^2 dz
CREDIBLE_LEVEL = 0.95
MASTER_SEED = 2026
REPORT_EVERY = 25
OVERWRITE = False
GEB_M = 32
ALPHA = 1.0


## Imports and data generation


In [ ]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from experiment_utils.transforms import apply_normal_cdf, fit_normal_cdf

from methods.GEB import GEBTree

def sample_laplace(n, d, rng):
    return rng.laplace(loc=0., scale=1., size=(n, d))



## Result summaries and saving


In [ ]:
SUMMARY_COLUMNS = [
    'posterior_mean', 'posterior_sd', 'bias', 'interval_length', 'ci_lower', 'ci_upper', 'covered',
    'data_time_seconds', 'fit_time_seconds',
    'posterior_time_seconds', 'fit_and_posterior_time_seconds', 'total_time_seconds',
]

def add_summary_rows(table):
    mean_row = {column: np.nan for column in table.columns}
    se_row = {column: np.nan for column in table.columns}
    mean_row['repetition'], se_row['repetition'] = 'Mean', 'Standard error'
    for column in SUMMARY_COLUMNS:
        values = pd.to_numeric(table[column], errors='coerce')
        if column in {'fit_time_seconds', 'fit_and_posterior_time_seconds', 'total_time_seconds'}:
            values = values.loc[table['repetition'] != 1]
        values = values.dropna()
        if len(values):
            mean_row[column] = values.mean()
        if len(values) > 1:
            se_row[column] = values.std(ddof=1) / np.sqrt(len(values))
    return pd.concat([table, pd.DataFrame([mean_row, se_row])], ignore_index=True)

def atomic_write_csv(table, path):
    temporary = path.with_suffix(path.suffix + '.tmp')
    output = add_summary_rows(table)
    output.to_csv(temporary, index=False)
    temporary.replace(path)

## Saved results and resuming runs


In [ ]:
OUTPUT = PROJECT_ROOT / 'results' / 'posterior_uncertainty' / 'laplace_isd_coverage.csv'
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
COLUMNS = [
    'repetition', 'data_seed', 'posterior_seed', 'n_train', 'posterior_draws',
    'geb_alpha', 'geb_m', 'credible_level', 'integration', 'n_leaves', 'true_isd',
    'posterior_mean', 'posterior_sd', 'bias', 'interval_length', 'ci_lower',
    'ci_upper', 'covered', 'data_time_seconds', 'fit_time_seconds',
    'posterior_time_seconds', 'fit_and_posterior_time_seconds', 'total_time_seconds',
]
if OUTPUT.exists() and not OVERWRITE:
    results = pd.read_csv(OUTPUT)
    if set(results.columns) != set(COLUMNS):
        raise ValueError('Incompatible checkpoint. Clear the old GEB ISD results before rerunning.')
    numeric_repetition = pd.to_numeric(results['repetition'], errors='coerce')
    results = results.loc[numeric_repetition.notna()].copy()
    results['repetition'] = numeric_repetition.loc[numeric_repetition.notna()].astype(int)
    for column in COLUMNS:
        if column not in {'repetition', 'covered', 'integration'}:
            results[column] = pd.to_numeric(results[column], errors='raise')
    results['covered'] = (results['ci_lower'] <= TRUE_ISD) & (TRUE_ISD <= results['ci_upper'])
    expected_settings = {
        'n_train': N_TRAIN, 'posterior_draws': POSTERIOR_DRAWS,
        'geb_alpha': ALPHA, 'geb_m': GEB_M, 'credible_level': CREDIBLE_LEVEL,
        'integration': 'analytic', 'true_isd': TRUE_ISD,
    }
    for column, expected in expected_settings.items():
        if not (results[column] == expected).all():
            raise ValueError(f'Stale checkpoint: {column} does not match the current settings.')
    if not (results.data_seed == MASTER_SEED + results.repetition - 1).all():
        raise ValueError('Stale checkpoint: data seeds do not match.')
    if not (results.posterior_seed == MASTER_SEED + 1_000_000 + results.repetition - 1).all():
        raise ValueError('Stale checkpoint: posterior seeds do not match.')
else:
    results = pd.DataFrame(columns=COLUMNS)
atomic_write_csv(results, OUTPUT)
print(f'PPB concentration alpha = {ALPHA:g}')
print(f'true ISD = {TRUE_ISD:.10g} (exact)')
print(f'output = {OUTPUT}')


## Run coverage experiment

In [ ]:
tail = (1. - CREDIBLE_LEVEL) / 2.
completed = set(results['repetition'].astype(int)) if len(results) else set()
for repetition in range(1, N_REPETITIONS + 1):
    if repetition in completed:
        continue
    total_start = time.perf_counter()
    data_seed = MASTER_SEED + repetition - 1
    start = time.perf_counter()
    x_train = sample_laplace(N_TRAIN, DIMENSION, np.random.default_rng(data_seed))
    data_time = time.perf_counter() - start

    parameters = fit_normal_cdf(x_train)
    u_train = apply_normal_cdf(x_train, parameters)
    start = time.perf_counter()
    tree = GEBTree(
        DIMENSION, np.zeros(DIMENSION), np.ones(DIMENSION),
        ALPHA, u_train, grid=True, m=GEB_M,
    )
    fit_time = time.perf_counter() - start

    posterior_seed = MASTER_SEED + 1_000_000 + repetition - 1
    start = time.perf_counter()
    draws = tree.sample_integrated_squared_density(
        POSTERIOR_DRAWS, normal_sd=parameters[1], random_state=posterior_seed,
    )
    lower, upper = np.quantile(draws, [tail, 1. - tail])
    n_leaves = len(tree.collect_leaves()[0])
    posterior_time = time.perf_counter() - start

    row = {
        'repetition': repetition, 'data_seed': data_seed,
        'posterior_seed': posterior_seed, 'n_train': N_TRAIN,
        'posterior_draws': POSTERIOR_DRAWS,
        'geb_alpha': ALPHA, 'geb_m': GEB_M, 'credible_level': CREDIBLE_LEVEL,
        'integration': 'analytic', 'n_leaves': n_leaves,
        'true_isd': TRUE_ISD,
        'posterior_mean': draws.mean(), 'posterior_sd': draws.std(ddof=1),
        'bias': draws.mean() - TRUE_ISD, 'interval_length': upper - lower,
        'ci_lower': lower, 'ci_upper': upper,
        'covered': lower <= TRUE_ISD <= upper,
        'data_time_seconds': data_time, 'fit_time_seconds': fit_time,
        'posterior_time_seconds': posterior_time,
        'fit_and_posterior_time_seconds': fit_time + posterior_time,
        'total_time_seconds': time.perf_counter() - total_start,
    }
    results = pd.concat([results, pd.DataFrame([row])], ignore_index=True)
    results = results.sort_values('repetition')
    atomic_write_csv(results, OUTPUT)
    if repetition == 1 or repetition % REPORT_EVERY == 0:
        coverage = results['covered'].astype(bool).mean()
        print(f'{repetition:04d}: coverage={coverage:.3f}, time={row["total_time_seconds"]:.3f}s')

## Summary

In [ ]:
coverage = results['covered'].astype(bool).mean()
summary = {
    'truth': 'Laplace(0, 1)',
    'dimension': DIMENSION,
    'geb_alpha': ALPHA,
    'integration': 'analytic',
    'completed_repetitions': len(results),
    'true_isd_exact': TRUE_ISD,
    'coverage_rate': float(coverage),
    'coverage_monte_carlo_se': float(np.sqrt(coverage*(1.-coverage)/len(results))),
    'mean_interval_width': float((results.ci_upper-results.ci_lower).mean()),
    'mean_bias': float(results.bias.mean()),
    'mean_total_time_seconds': float(results.loc[results.repetition != 1, 'total_time_seconds'].mean()),
    'se_total_time_seconds': float(results.loc[results.repetition != 1, 'total_time_seconds'].sem()),
    'mean_fit_and_posterior_time_seconds': float((results.loc[results.repetition != 1, 'fit_time_seconds'] + results.loc[results.repetition != 1, 'posterior_time_seconds']).mean()),
    'se_fit_and_posterior_time_seconds': float(results.loc[results.repetition != 1, 'fit_and_posterior_time_seconds'].sem()),
    'total_compute_time_seconds': float(results.total_time_seconds.sum()),
}
pd.DataFrame([summary]).to_csv(OUTPUT.with_name('laplace_isd_coverage_geb_summary.csv'), index=False)
summary

## Posterior comparison on a shared dataset

Read the dataset and DPMM ISD posterior draws saved by `02_posterior_uncertainty_dpmm.Rmd`. Fit GEB to the same dataset and plot both posterior distributions. The dashed line marks the true value, \(\theta(p)=1/4\).


In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from IPython.display import display

COMPARISON_DIR = PROJECT_ROOT / 'results' / 'posterior_uncertainty'
R_SAMPLE_FILE = COMPARISON_DIR / 'x_train.csv'
R_DRAWS_FILE = COMPARISON_DIR / 'dpmm_draws.csv'
R_SETTINGS_FILE = COMPARISON_DIR / 'posterior_comparison_settings.csv'

missing = [path for path in (R_SAMPLE_FILE, R_DRAWS_FILE, R_SETTINGS_FILE) if not path.exists()]
if missing:
    missing_text = '\n'.join(f'  - {path}' for path in missing)
    raise FileNotFoundError(
        'The R comparison outputs are missing. Run 02_posterior_uncertainty_dpmm.Rmd '
        f'from top to bottom first.\n{missing_text}'
    )

comparison_settings = pd.read_csv(R_SETTINGS_FILE).iloc[0]
expected_settings = {
    'data_seed': 2034,
    'mcmc_seed': 2_002_026,
    'n_train': N_TRAIN,
    'posterior_draws': POSTERIOR_DRAWS,
    'burn_in': 1_000,
    'integration': 'analytic',
}
for name, expected in expected_settings.items():
    observed = comparison_settings[name]
    if observed != expected:
        raise ValueError(f'R comparison setting {name}={observed}; expected {expected}.')

comparison_x = pd.read_csv(R_SAMPLE_FILE)['x'].to_numpy(float).reshape(-1, 1)
dpmm_draws = pd.read_csv(R_DRAWS_FILE)['isd'].to_numpy(float)
if comparison_x.shape != (N_TRAIN, 1):
    raise ValueError(f'Unexpected representative sample shape: {comparison_x.shape}')
if len(dpmm_draws) != POSTERIOR_DRAWS or not np.all(np.isfinite(dpmm_draws)):
    raise ValueError('The saved DPMM posterior draws are incomplete or non-finite.')

comparison_parameters = fit_normal_cdf(comparison_x)
comparison_u = apply_normal_cdf(comparison_x, comparison_parameters)
comparison_tree = GEBTree(
    DIMENSION, np.zeros(DIMENSION), np.ones(DIMENSION),
    ALPHA, comparison_u, grid=True, m=GEB_M,
)
geb_draws = comparison_tree.sample_integrated_squared_density(
    POSTERIOR_DRAWS,
    normal_sd=comparison_parameters[1],
    random_state=1_002_026,
)
pd.DataFrame({'isd': geb_draws}).to_csv(COMPARISON_DIR / 'geb_draws.csv', index=False)

print(f'Representative sample: n={len(comparison_x)}, data seed={expected_settings["data_seed"]}')
print(f'GEB: m={GEB_M}, alpha={ALPHA:g}, draws={len(geb_draws)}')
print(f'DPMM draws loaded from R: {len(dpmm_draws)}')


In [ ]:
def nrd0_bandwidth(values):
    values = np.asarray(values, dtype=float)
    sd = values.std(ddof=1)
    iqr = np.subtract(*np.percentile(values, [75, 25]))
    scale = min(sd, iqr / 1.34) if iqr > 0 else sd
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError('Cannot determine a positive KDE bandwidth.')
    return 0.9 * scale * len(values) ** (-0.2)

all_draws = np.concatenate([geb_draws, dpmm_draws])
span = np.ptp(all_draws)
padding = 0.08 * span if span > 0 else 0.01
x_grid = np.linspace(all_draws.min() - padding, all_draws.max() + padding, 1024)
common_bandwidth = 1.5 * nrd0_bandwidth(all_draws)

def kde_with_absolute_bandwidth(values, grid, bandwidth):
    values = np.asarray(values, dtype=float)
    kde = gaussian_kde(values)
    kde.set_bandwidth(bw_method=bandwidth / values.std(ddof=1))
    return kde(grid)

geb_density = kde_with_absolute_bandwidth(geb_draws, x_grid, common_bandwidth)
dpmm_density = kde_with_absolute_bandwidth(dpmm_draws, x_grid, common_bandwidth)

BLUE, RED, TRUTH = '#4C78A8', '#D95F5F', '#222222'
fig, ax = plt.subplots(figsize=(7.2, 3.2))
ax.plot(x_grid, dpmm_density, color=RED, linewidth=2.4, label='DPMM')
ax.plot(x_grid, geb_density, color=BLUE, linewidth=2.4, label='PPB + GEB')
ax.axvline(TRUE_ISD, color=TRUTH, linestyle='--', linewidth=2.2,
           label=r'$\theta(p)=1/4$')
ax.set_xlabel(r'Integrated squared density $\theta$', fontsize=13)
ax.set_ylabel('Posterior density', fontsize=13)
ax.set_ylim(bottom=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(labelsize=11)
ax.legend(frameon=False, fontsize=12)
fig.tight_layout()

FIGURE_FILE = COMPARISON_DIR / 'laplace_isd_posterior_comparison.pdf'
fig.savefig(FIGURE_FILE, bbox_inches='tight')
display(fig)
plt.close(fig)
print(f'Saved posterior figure to {FIGURE_FILE}')


## Result location

Coverage results, posterior draws, and the comparison figure are saved in `results/posterior_uncertainty/`.


In [ ]:
print(PROJECT_ROOT / 'results' / 'posterior_uncertainty')
